Write a function that accepts a square matrix (N x N 2D array) and returns the determinant of the matrix.

How to take the determinant of a matrix -- it is simplest to start with the smallest cases:

A 1x1 matrix |a| has determinant a.

A 2x2 matrix [ [a, b], [c, d] ] or

|a  b|
|c  d|
has determinant: a*d - b*c.

The determinant of an n x n sized matrix is calculated by reducing the problem to the calculation of the determinants of n matrices ofn-1 x n-1 size.

For the 3x3 case, [ [a, b, c], [d, e, f], [g, h, i] ] or

|a b c|  
|d e f|  
|g h i|  
the determinant is: a * det(a_minor) - b * det(b_minor) + c * det(c_minor) where det(a_minor) refers to taking the determinant of the 2x2 matrix created by crossing out the row and column in which the element a occurs:

|- - -|
|- e f|
|- h i|  
Note the alternation of signs.

The determinant of larger matrices are calculated analogously, e.g. if M is a 4x4 matrix with first row [a, b, c, d], then:

det(M) = a * det(a_minor) - b * det(b_minor) + c * det(c_minor) - d * det(d_minor)

Ideas.

Background info: The formula given here is the Laplace expansion for determinants. Determinants rarely ever use in numerical linear algebra, but I've never written an algorithm to compute it. Let's go.

* As we can carry out Laplace expansion on any $i$-th row, or any $j$th column, we will standardise, and do this for the $0$-th row.
* We can now write the Laplace expansion of the $0$-th row of an $n \times n$ array as,
$$\det(A) = \sum^{n-1}_{j=0} (-1)^j a_{0j} \cdot M_{0, j}.$$
* Because we always Laplace expand along $0$th row, refer to $M_{0, j}$ as the $j$-th minor.
* Write a function that takes an $n \times n$ array and generates an $n$ lots of $(n-1) \times (n-1)$ arrays, and the $n$ coefficients that will be used in a Laplace expansion of the $n \times n$ matrix, using $3 \times 3$ array as main test case.


In [66]:
arr = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

coeff_coord = [0, 0]

# Construct a corresponding list of indices - this may be useful.

coords = [[i, j] for i in range(len(arr)) for j in range(len(arr))]
coords

[[0, 0], [0, 1], [0, 2], [1, 0], [1, 1], [1, 2], [2, 0], [2, 1], [2, 2]]

In [34]:
# Now we can construct a list of minor matrix indices by filtering this.

minor = [arr[i][j] for i in range(len(arr)) for j in range(len(arr)) if i != coeff_coord[0] and j != coeff_coord[1]]
minor

[5, 6, 8, 9]

In [ ]:
# Constructing a list of indices and picking it out like you do in a list comprehension results in a 2D list that does not preserve structure, which you need to manually reshape back.
# Instead, use slicing and concatenating, iterating over rows. Here is one example of the logic:

i = 0
j = 2
row = [4, 5, 6]
row[:j] + row[j+1:]

In [45]:
jth_minor = [row[:j]+ row[j+1:] for row in arr[1:]]
minor

[[4, 5], [7, 8]]

In [78]:
arr_size = len(arr)
signed_coeffs = [((-1) ** j) * arr[0][j] for j in range(0, arr_size)]

minors = []
for j in range(0, arr_size):
    jth_minor = [row[:j] + row[j+1:] for row in arr[1:]]
    minors.append(jth_minor)
print(signed_coeffs, minors)

[1, -2, 3] [[[5, 6], [8, 9]], [[4, 6], [7, 9]], [[4, 5], [7, 8]]]


In [80]:
# Remember the model - slicing lets you put things in between.

for minor in minors:
    minor_size = len(minor)
    signed_minor_coeffs = [((-1) ** j) * minor[0][j] for j in range(0, minor_size)]
    sub_minors = []
    for j in range(0, minor_size):
        jth_sub_minor = [row[:j] + row[j+1:] for row in minor[1:]]
        sub_minors.append(jth_sub_minor)

print(signed_minor_coeffs, sub_minors)

[4, -5] [[[8]], [[7]]]


In [154]:
# Do not keep track of the signed coefficients in a global coefficient list, use recursion.

arr = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

i, j = 0, 0

signed_coeff = ((-1) ** 0) * arr[0][0]
jth_minor = [row[:j] + row[j+1:] for row in arr[1:]]

In [ ]:
def cofactor(arr, index):
    arr_size = len(arr)

    if not arr_size:
        return arr[0]
    else:
        i, j = index[0], index[1]
        jth_minor = [row[:j] + row[j+1:] for row in arr[1:]]
        # Problem is here.
        signed_coeff = ((-1) ** (i + j)) * jth_minor[i][j]

    return signed_coeff, jth_minor

In [184]:
cofactor(arr, (0, 0))

(5, [[5, 6], [8, 9]])

In [153]:
cofactor([[5, 6], [8, 9]], (0, 0))

(9, [[9]])

In [182]:
# Now write the loop.

arr = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
coords = coords = [(i, j) for i in range(len(arr)) for j in range(len(arr))]
arr_size = len(arr)

if arr_size == 1:
    print(arr[0])

index = (0,2)
i = index[0]
j = index[1]

coeff_list = [((-1) ** (i + j)) * arr[i][j]]
jth_minor = arr

print(coeff_list, jth_minor)

coeff, jth_minor = cofactor(jth_minor, index)
coeff_list.append(coeff)
print(coeff_list, jth_minor)

coeff, jth_minor = cofactor(jth_minor, index)
coeff_list.append(coeff)
print(coeff_list, jth_minor)


[3] [[1, 2, 3], [4, 5, 6], [7, 8, 9]]


IndexError: list index out of range

In [178]:
arr = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
arr_size = len(arr)
if arr_size == 1:
    print(arr[0])

coords = coords = [(i, j) for i in range(arr_size) for j in range(arr_size)]

index = (0,1)
i = index[0]
j = index[1]

coeff_list = [((-1) ** (i + j)) * arr[i][j]]
jth_minor = arr

for i in range(0, arr_size-1):
    coeff, jth_minor = cofactor(jth_minor, index)
    coeff_list.append(coeff)

product = 1
for coeff in coeff_list:
    product *= coeff

IndexError: list index out of range